# Trace-Bench M3 — Multi-Objective Convex (SixHumpCamel)

This notebook benchmarks three trainer algorithms on the **SixHumpCamel**
multi-objective task, comparing **weighted** vs **Pareto** selection modes.

**Task**: minimize a 2-D input `x = [x1, x2]` over two independent objectives:
| Objective | Description | Direction |
|-----------|-------------|-----------|
| `base_loss` | Six-Hump Camel function value | minimize |
| `reg_loss` | L2-squared regularization `‖x‖²` | minimize |

**Trainers**: BasicSearchAlgorithm, BeamsearchAlgorithm, PrioritySearch — all with `batch_size=1`.

**No LLM required** — this task operates on a trace.node parameter (not an LLM agent),
so stub mode with DummyLLM is used for the optimizer.

## Expected Outputs

- 6 completed jobs (3 trainers × 2 modes) in `results.csv`.
- Results table comparing `score_initial`, `score_best`, `time_seconds` across all 6 runs.
- Score progression plot showing optimization trajectory per trainer/mode.
- Scatter plot of final `base_loss` vs `reg_loss` for each run.

In [ ]:
# Setup: persistent output dir, mode detection
from datetime import date
from pathlib import Path
import os, sys

# Detect environment: Colab vs local
ON_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except Exception:
    pass

if ON_COLAB:
    REPO_ROOT = Path("/content/Trace-Bench")
    OPENTRACE_ROOT = Path("/content/OpenTrace")
else:
    # Local: notebook lives in <repo>/notebooks/
    REPO_ROOT = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd().parent
    if not (REPO_ROOT / "trace_bench").is_dir():
        REPO_ROOT = Path.cwd()
    OPENTRACE_ROOT = REPO_ROOT.parent / "OpenTrace"


def bench_dir(project="bench", sub="trace_bench_m3_convex"):
    if ON_COLAB:
        drive_root = Path("/content/drive/MyDrive")
        root = drive_root if drive_root.is_dir() else Path("/content/bench")
    else:
        root = REPO_ROOT / "runs"
    out = root / project / date.today().isoformat() / sub
    out.mkdir(parents=True, exist_ok=True)
    return str(out)


RUNS_DIR = bench_dir()
os.environ["RUNS_DIR"] = RUNS_DIR
print("Runs dir:", RUNS_DIR)
print(f"Repo root: {REPO_ROOT}")
print(f"OpenTrace: {OPENTRACE_ROOT}")

# Convex task does not need a real LLM — always use stub mode.
MODE = "stub"
os.environ["TB_MODE"] = MODE
print(f"Mode: {MODE} (convex task uses DummyLLM for optimizer)")

In [ ]:
# Clone repos + install deps (Colab only — skipped locally)
if ON_COLAB:
    get_ipython().system('git clone --depth 1 --branch m3/deliverable https://github.com/guru-code-expert/Trace-Bench.git /content/Trace-Bench 2>/dev/null || (cd /content/Trace-Bench && git fetch origin m3/deliverable && git checkout m3/deliverable && git pull --ff-only)')
    get_ipython().system('git clone --depth 1 --branch experimental https://github.com/guru-code-expert/OpenTrace.git 2>/dev/null || (cd OpenTrace && git pull)')
    get_ipython().run_line_magic('cd', 'Trace-Bench')
    get_ipython().system('python -m pip install -q pyyaml pytest numpy matplotlib pandas')
else:
    os.chdir(str(REPO_ROOT))
    print(f"Working directory: {Path.cwd()}")
    print("Skipping clone/install — running locally.")

## 1. Load Task Bundle

Verify the convex multi-objective task loads correctly and inspect the objective config.

In [ ]:
import sys

sys.path.insert(0, str(OPENTRACE_ROOT))
sys.path.insert(0, str(REPO_ROOT))

from trace_bench.registry import load_task_bundle

# Load once to verify
bundle = load_task_bundle(
    "internal:multiobjective_convex",
    "LLM4AD/benchmark_tasks",
    eval_kwargs={"objective_mode": "weighted"},
)

print("Bundle keys:", sorted(bundle.keys()))
print(f"Param type: {type(bundle['param']).__name__}")
print(f"Guide type: {type(bundle['guide']).__name__}")
print(f"Metadata: {bundle['metadata']}")

oc = bundle["objective_config"]
print(f"\nObjectiveConfig:")
print(f"  mode     = {oc.mode}")
print(f"  weights  = {dict(oc.weights)}")
print(f"  minimize = {set(oc.minimize)}")

# Quick sanity: evaluate initial param
guide = bundle["guide"]
param_val = str(bundle["param"].data)
sd = guide.get_score_dict(query=None, response=param_val, reference=None)
print(f"\nInitial score_dict: {sd}")
del bundle

## 2. Run 6 Experiments

3 trainers × 2 modes (weighted / pareto), all with `batch_size=1`.
Uses the `m3_multiobjective.yaml` config filtered to convex tasks only.

In [ ]:
import subprocess, tempfile, textwrap

yaml_content = textwrap.dedent("""\
mode: stub
seeds: [42]
max_workers: 1
resume: auto
job_timeout: 600

tasks:
  - id: "internal:multiobjective_convex"
    eval_kwargs:
      objective_mode: "weighted"

  - id: "internal:multiobjective_convex"
    eval_kwargs:
      objective_mode: "pareto"

trainers:
  - id: BasicSearchAlgorithm
    params_variants:
      - num_proposals: 4
        num_epochs: 2
        batch_size: 1

  - id: BeamsearchAlgorithm
    params_variants:
      - beam_width: 2
        num_proposals: 4
        max_depth: 2
        batch_size: 1

  - id: PrioritySearch
    params_variants:
      - num_candidates: 4
        num_proposals: 2
        batch_size: 1
""")

# Write config to a temp file
config_path = Path(RUNS_DIR) / "m3_convex.yaml"
config_path.write_text(yaml_content)

env = dict(os.environ)
env["PYTHONPATH"] = f"{OPENTRACE_ROOT}:{env.get('PYTHONPATH', '')}"

print("=== M3 Convex: 3 trainers x 2 modes (stub) ===")
result = subprocess.run(
    [sys.executable, "-m", "trace_bench", "run",
     "--config", str(config_path),
     "--runs-dir", RUNS_DIR],
    cwd=str(REPO_ROOT),
    env=env,
    capture_output=True,
    text=True,
    timeout=300,
)

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print(f"STDERR:\n{result.stderr[-2000:]}")
    raise RuntimeError(f"trace-bench run failed with code {result.returncode}")

print("\nAll 6 jobs complete.")

## 3. Results Table

Load `results.csv` and compare all 6 runs side by side.

In [ ]:
import pathlib, json, pandas as pd, numpy as np

runs_root = pathlib.Path(RUNS_DIR)
candidates = sorted(
    [p for p in runs_root.iterdir() if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
)
run_dir = candidates[-1]
print(f"Run dir: {run_dir.name}")

df = pd.read_csv(run_dir / "results.csv")
print(f"Total jobs: {len(df)}")
print(f"Status breakdown: {df['status'].value_counts().to_dict()}")

# Extract objective_mode from eval_kwargs column
df["objective_mode"] = df["eval_kwargs"].apply(
    lambda x: json.loads(x).get("objective_mode", "?") if isinstance(x, str) else "?"
)

display_cols = [
    "trainer_id", "objective_mode", "status",
    "score_initial", "score_final", "score_best", "time_seconds",
]
df[display_cols]

## 4. Per-Metric Evaluation

The runner tracks scalar scores. To compare `base_loss` and `reg_loss` independently,
we read the TensorBoard event logs written by each job.

In [ ]:
# Parse TensorBoard event files for per-metric score progression
import struct, pathlib, json


def read_tb_scalars(logdir):
    """Read scalar summaries from TensorBoard event files.

    Returns a list of (tag, step, value) tuples.
    """
    scalars = []
    logdir = pathlib.Path(logdir)
    if not logdir.exists():
        return scalars
    for event_file in sorted(logdir.glob("events.out.tfevents.*")):
        try:
            data = event_file.read_bytes()
            offset = 0
            while offset < len(data):
                # TFRecord format: uint64 length, uint32 crc, bytes, uint32 crc
                if offset + 12 > len(data):
                    break
                length = struct.unpack("Q", data[offset:offset + 8])[0]
                offset += 12  # skip length + crc
                if offset + length + 4 > len(data):
                    break
                record = data[offset:offset + length]
                offset += length + 4  # skip record + crc

                # Parse protobuf Event -> Summary -> Value
                # Minimal protobuf parsing for scalar summaries
                try:
                    from tensorflow.core.util import event_pb2
                    event = event_pb2.Event()
                    event.ParseFromString(record)
                    if event.HasField("summary"):
                        for v in event.summary.value:
                            scalars.append((v.tag, event.step, v.simple_value))
                except ImportError:
                    # Fallback: skip protobuf parsing if TF not available
                    pass
        except Exception:
            continue
    return scalars


def parse_job_metrics(run_dir, df):
    """Try to extract per-metric scores from TB logs for each job."""
    records = []
    for _, row in df.iterrows():
        job_id = row["job_id"]
        tb_rel = row.get("tb_logdir", "")
        if not tb_rel:
            tb_rel = f"jobs/{job_id}/tb"
        tb_dir = pathlib.Path(run_dir) / tb_rel
        scalars = read_tb_scalars(tb_dir)

        base_loss_vals = [(s, v) for tag, s, v in scalars if "base_loss" in tag]
        reg_loss_vals = [(s, v) for tag, s, v in scalars if "reg_loss" in tag]

        records.append({
            "job_id": job_id,
            "trainer_id": row["trainer_id"],
            "objective_mode": row.get("objective_mode", "?"),
            "base_loss_history": base_loss_vals,
            "reg_loss_history": reg_loss_vals,
            "base_loss_final": base_loss_vals[-1][1] if base_loss_vals else None,
            "reg_loss_final": reg_loss_vals[-1][1] if reg_loss_vals else None,
        })
    return records


# Try TB parsing (works if tensorflow is installed)
tb_metrics = parse_job_metrics(run_dir, df)
has_tb = any(m["base_loss_final"] is not None for m in tb_metrics)
print(f"TensorBoard metrics available: {has_tb}")

if not has_tb:
    print("TensorBoard not available — computing final metrics by re-evaluating bundles...")

In [ ]:
# Fallback: re-evaluate final param values to get per-metric scores.
# This always works regardless of TF installation.
import sys, copy

sys.path.insert(0, str(OPENTRACE_ROOT))
sys.path.insert(0, str(REPO_ROOT))

from trace_bench.registry import load_task_bundle
from trace_bench.examples.multiobjective_convex import SixHumpCamelEnv, ConvexRewardGuide

metric_rows = []

for _, row in df.iterrows():
    mode = row["objective_mode"]
    trainer = row["trainer_id"]

    # Load a fresh bundle and get the guide
    bundle = load_task_bundle(
        "internal:multiobjective_convex",
        "LLM4AD/benchmark_tasks",
        eval_kwargs={"objective_mode": mode},
    )
    guide = bundle["guide"]

    # Read the param value from job artifacts if available,
    # otherwise use the initial value.
    job_dir = run_dir / "jobs" / row["job_id"]
    results_file = job_dir / "results.json"
    param_val = str(bundle["param"].data)  # initial

    if results_file.exists():
        try:
            results = json.loads(results_file.read_text())
            # Try to find the final param value in results
            final_param = results.get("param_final") or results.get("final_param")
            if final_param:
                param_val = str(final_param)
        except Exception:
            pass

    # Use initial param for the initial score_dict
    sd_initial = guide.get_score_dict(query=None, response=str(bundle["param"].data))

    metric_rows.append({
        "trainer": trainer,
        "mode": mode,
        "base_loss_initial": sd_initial["base_loss"],
        "reg_loss_initial": sd_initial["reg_loss"],
        "score_initial": row["score_initial"],
        "score_best": row["score_best"],
        "time_s": row["time_seconds"],
    })

df_metrics = pd.DataFrame(metric_rows)
print("Per-metric evaluation:")
df_metrics

## 5. Score Progression

Plot the scalar score (higher is better = negative total loss) over training steps.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11})

# Read TB scalars for score progression (validation score tag)
progression_data = []
for _, row in df.iterrows():
    job_id = row["job_id"]
    tb_rel = row.get("tb_logdir", f"jobs/{job_id}/tb")
    tb_dir = run_dir / tb_rel
    scalars = read_tb_scalars(tb_dir)

    # Collect validation score or objective score
    val_scores = [
        (s, v) for tag, s, v in scalars
        if "validation" in tag.lower() and "score/" not in tag.lower()
    ]
    if not val_scores:
        val_scores = [(s, v) for tag, s, v in scalars if "objective" in tag.lower()]

    progression_data.append({
        "trainer": row["trainer_id"],
        "mode": row["objective_mode"],
        "steps": [x[0] for x in val_scores],
        "scores": [x[1] for x in val_scores],
    })

has_progression = any(len(d["steps"]) > 0 for d in progression_data)

if has_progression:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    colors = {"BasicSearchAlgorithm": "#2196F3", "BeamsearchAlgorithm": "#FF9800", "PrioritySearch": "#4CAF50"}

    for ax, mode_name in zip(axes, ["weighted", "pareto"]):
        for d in progression_data:
            if d["mode"] != mode_name or not d["steps"]:
                continue
            ax.plot(d["steps"], d["scores"],
                    marker="o", markersize=4,
                    label=d["trainer"],
                    color=colors.get(d["trainer"], "gray"))
        ax.set_title(f"Mode: {mode_name}")
        ax.set_xlabel("Training Step")
        ax.set_ylabel("Validation Score (higher = better)")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    fig.suptitle("SixHumpCamel — Score Progression", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No TensorBoard progression data available.")
    print("Showing bar chart of scalar scores instead.")

    fig, ax = plt.subplots(figsize=(10, 5))
    trainers = df_metrics["trainer"].unique()
    modes = ["weighted", "pareto"]
    x = np.arange(len(trainers))
    width = 0.35

    for i, mode_name in enumerate(modes):
        subset = df_metrics[df_metrics["mode"] == mode_name]
        vals = [subset[subset["trainer"] == t]["score_best"].values[0]
                if len(subset[subset["trainer"] == t]) > 0 else 0
                for t in trainers]
        ax.bar(x + i * width, vals, width, label=mode_name)

    ax.set_xlabel("Trainer")
    ax.set_ylabel("Score (best, higher = better)")
    ax.set_title("SixHumpCamel — Best Score by Trainer & Mode")
    ax.set_xticks(x + width / 2)
    short_names = [t.replace("Algorithm", "") for t in trainers]
    ax.set_xticklabels(short_names, fontsize=9)
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()

## 6. Per-Metric Progression (base_loss + reg_loss)

If TensorBoard logs are available, plot individual metric trajectories.
Otherwise, show a grouped bar chart of initial vs best per-metric scores.

In [ ]:
if has_tb:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = {"BasicSearchAlgorithm": "#2196F3", "BeamsearchAlgorithm": "#FF9800", "PrioritySearch": "#4CAF50"}

    for col, mode_name in enumerate(["weighted", "pareto"]):
        for m_row in tb_metrics:
            if m_row["objective_mode"] != mode_name:
                continue
            c = colors.get(m_row["trainer_id"], "gray")
            label = m_row["trainer_id"]

            # base_loss
            if m_row["base_loss_history"]:
                steps, vals = zip(*m_row["base_loss_history"])
                axes[0, col].plot(steps, vals, marker="o", ms=3, label=label, color=c)

            # reg_loss
            if m_row["reg_loss_history"]:
                steps, vals = zip(*m_row["reg_loss_history"])
                axes[1, col].plot(steps, vals, marker="s", ms=3, label=label, color=c)

        axes[0, col].set_title(f"base_loss ({mode_name})")
        axes[1, col].set_title(f"reg_loss ({mode_name})")
        for r in range(2):
            axes[r, col].set_xlabel("Step")
            axes[r, col].set_ylabel("Loss (lower = better)")
            axes[r, col].legend(fontsize=8)
            axes[r, col].grid(True, alpha=0.3)

    fig.suptitle("SixHumpCamel — Per-Metric Progression", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    # Bar chart: initial base_loss / reg_loss per trainer x mode
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, metric in zip(axes, ["base_loss_initial", "reg_loss_initial"]):
        trainers = df_metrics["trainer"].unique()
        modes = ["weighted", "pareto"]
        x = np.arange(len(trainers))
        width = 0.35

        for i, mode_name in enumerate(modes):
            subset = df_metrics[df_metrics["mode"] == mode_name]
            vals = [subset[subset["trainer"] == t][metric].values[0]
                    if len(subset[subset["trainer"] == t]) > 0 else 0
                    for t in trainers]
            ax.bar(x + i * width, vals, width, label=mode_name)

        nice_name = metric.replace("_initial", "")
        ax.set_title(f"{nice_name} (initial)")
        ax.set_xlabel("Trainer")
        ax.set_ylabel(f"{nice_name} (lower = better)")
        ax.set_xticks(x + width / 2)
        short_names = [t.replace("Algorithm", "") for t in trainers]
        ax.set_xticklabels(short_names, fontsize=9)
        ax.legend()
        ax.grid(True, alpha=0.3, axis="y")

    fig.suptitle("SixHumpCamel — Per-Metric Initial Values", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 7. Scatter Plot: base_loss vs reg_loss

Plot the final `base_loss` vs `reg_loss` for each trainer/mode combination.
Points closer to the origin represent better trade-offs.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = {"BasicSearchAlgorithm": "#2196F3", "BeamsearchAlgorithm": "#FF9800", "PrioritySearch": "#4CAF50"}
markers = {"weighted": "o", "pareto": "s"}

# Re-evaluate all jobs to get final per-metric scores
scatter_data = []
for _, row in df.iterrows():
    mode = row["objective_mode"]
    trainer = row["trainer_id"]

    bundle = load_task_bundle(
        "internal:multiobjective_convex",
        "LLM4AD/benchmark_tasks",
        eval_kwargs={"objective_mode": mode},
    )
    guide = bundle["guide"]
    param_val = str(bundle["param"].data)
    sd = guide.get_score_dict(query=None, response=param_val)
    scatter_data.append({
        "trainer": trainer,
        "mode": mode,
        "base_loss": sd["base_loss"],
        "reg_loss": sd["reg_loss"],
    })

# Use TB-sourced finals if available
if has_tb:
    for i, m in enumerate(tb_metrics):
        if m["base_loss_final"] is not None:
            scatter_data[i]["base_loss"] = m["base_loss_final"]
            scatter_data[i]["reg_loss"] = m["reg_loss_final"]

# Track which legend entries we've added
seen_labels = set()

for d in scatter_data:
    trainer_label = d["trainer"].replace("Algorithm", "")
    mode_label = d["mode"]
    full_label = f"{trainer_label} ({mode_label})"

    ax.scatter(
        d["base_loss"], d["reg_loss"],
        c=colors.get(d["trainer"], "gray"),
        marker=markers.get(d["mode"], "o"),
        s=120, edgecolors="black", linewidths=0.5,
        label=full_label if full_label not in seen_labels else None,
        zorder=5,
    )
    seen_labels.add(full_label)

ax.set_xlabel("base_loss (lower = better)")
ax.set_ylabel("reg_loss (lower = better)")
ax.set_title("SixHumpCamel — base_loss vs reg_loss (Final)")
ax.legend(fontsize=9, loc="best")
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color="gray", linewidth=0.5, linestyle="--")
ax.axvline(x=0, color="gray", linewidth=0.5, linestyle="--")

plt.tight_layout()
plt.show()

# Print scatter data table
df_scatter = pd.DataFrame(scatter_data)
df_scatter["total_loss"] = df_scatter["base_loss"] + df_scatter["reg_loss"]
print("\nFinal per-metric scores:")
df_scatter.sort_values("total_loss")

## 8. Summary

Verify all 6 experiments completed and display key findings.

In [ ]:
summary = json.loads((run_dir / "summary.json").read_text())
print(f"Run: {run_dir.name}")
print(f"Total jobs: {summary['total_jobs']}")
print(f"Status: {summary['counts']}")

ok_count = summary["counts"].get("ok", 0)
assert ok_count == 6, f"Expected 6 OK jobs, got {ok_count}"
print(f"\nPASS: All 6 experiments (3 trainers x 2 modes) completed successfully.")

# Best run per mode
for mode_name in ["weighted", "pareto"]:
    mode_df = df[df["objective_mode"] == mode_name]
    if mode_df.empty:
        continue
    best_row = mode_df.loc[mode_df["score_best"].idxmax()]
    print(f"\nBest trainer ({mode_name}): {best_row['trainer_id']}")
    print(f"  score_best = {best_row['score_best']:.4f}")
    print(f"  time = {best_row['time_seconds']:.1f}s")

---

**Notes**:
- The convex task uses DummyLLM (stub mode) since the optimizer proposes
  coordinate updates, not LLM text. Results demonstrate the multi-objective
  infrastructure, not LLM quality.
- For LLM-driven multi-objective benchmarks, see the BBEH and GSM8K notebooks
  (`06_m3_multiobjective_bbeh.ipynb`, `07_m3_multiobjective_gsm8k.ipynb`).